# LCG 3DEP topographic profile

This notebook turns an ordered WGS 84 point table into a topographic profile by calling the deployed LCG elevation service. It demonstrates the client pattern that can later be used by ArcGIS Pro and the public web interface.

The workflow keeps point identity and order explicit, calculates distance on the WGS 84 ellipsoid, exposes `no_data` results, and writes reviewable CSV, GeoJSON, PNG, and metadata outputs. The returned elevations sample an approximately 10-meter bare-earth DEM and are not survey-grade values.

## 1. Environment setup

Before opening the notebook, activate the project Python 3.12 virtual environment and install the notebook dependency group from the repository root:

```powershell
python -m pip install -e ".[notebooks]"
python -m jupyter lab
```

Select that same virtual environment as the notebook kernel.

In [ ]:
from __future__ import annotations

import json
import math
import re
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import display
from pyproj import Geod

SERVICE_BASE_URL = "https://elevation.logiccloudgeo.com"
ELEVATION_UNITS = "feet"       # "feet" or "meters"
HORIZONTAL_UNITS = "miles"     # "miles" or "kilometers"
BATCH_SIZE = 250                  # API maximum is 500
REQUEST_TIMEOUT_SECONDS = 90
PAUSE_BETWEEN_BATCHES_SECONDS = 0.5

if ELEVATION_UNITS not in {"feet", "meters"}:
    raise ValueError("ELEVATION_UNITS must be 'feet' or 'meters'.")
if HORIZONTAL_UNITS not in {"miles", "kilometers"}:
    raise ValueError("HORIZONTAL_UNITS must be 'miles' or 'kilometers'.")
if not 1 <= BATCH_SIZE <= 500:
    raise ValueError("BATCH_SIZE must be between 1 and 500.")

## 2. Locate the input and output directories

Jupyter kernels can start in either the repository root or the `notebooks` directory. This cell searches upward for the project markers instead of assuming one current working directory. Change `INPUT_PATH` later to use an ArcGIS or project-specific CSV.

In [ ]:
def find_repository_root(start: Path) -> Path:
    candidates = (start.resolve(), *start.resolve().parents)
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "sample_data").is_dir():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Start JupyterLab from this project's repository."
    )


REPOSITORY_ROOT = find_repository_root(Path.cwd())
INPUT_PATH = REPOSITORY_ROOT / "sample_data" / "florida_profile_points.csv"
OUTPUT_DIRECTORY = REPOSITORY_ROOT / "output"

print(f"Repository: {REPOSITORY_ROOT}")
print(f"Input:      {INPUT_PATH}")
print(f"Output:     {OUTPUT_DIRECTORY}")

## 3. Check application health

The health route verifies DNS, TLS, App Service routing, the container, and FastAPI without consuming an upstream elevation query. A successful health check does not guarantee that the upstream elevation provider is available, but it separates application availability from provider availability.

In [ ]:
health_url = f"{SERVICE_BASE_URL.rstrip('/')}/health"
health_response = requests.get(health_url, timeout=15)
health_response.raise_for_status()
health = health_response.json()
display(pd.Series(health, name="value").to_frame())

## 4. Read the ordered point table

Required columns are `db_key`, `latitude`, and `longitude`. A numeric `sequence` column is recommended and controls distance order. Other columns, such as a source feature ID or label, are retained. If `sequence` is absent, the existing CSV row order is used.

The included Florida profile is synthetic test data, not a surveyed alignment.

In [ ]:
raw_points = pd.read_csv(INPUT_PATH)
print(f"Read {len(raw_points):,} input points.")
display(raw_points.head(10))

## 5. Validate locally before using the API

Client validation provides immediate row-specific feedback and avoids consuming network or provider capacity for a malformed request. The service independently validates every public request because it cannot trust clients to perform this step.

In [ ]:
REQUIRED_COLUMNS = {"db_key", "latitude", "longitude"}
DB_KEY_PATTERN = re.compile(r"^[A-Za-z0-9][A-Za-z0-9_.-]{0,63}$")


def validate_profile_points(input_frame: pd.DataFrame) -> pd.DataFrame:
    if input_frame.empty:
        raise ValueError("The profile table contains no points.")

    frame = input_frame.copy()
    frame.columns = [str(column).strip().lower() for column in frame.columns]

    missing = REQUIRED_COLUMNS - set(frame.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    frame["db_key"] = frame["db_key"].astype("string").str.strip()
    blank_keys = frame["db_key"].isna() | frame["db_key"].eq("")
    if blank_keys.any():
        raise ValueError(f"Blank db_key values at input rows {list(frame.index[blank_keys])}")

    invalid_keys = ~frame["db_key"].str.fullmatch(DB_KEY_PATTERN, na=False)
    if invalid_keys.any():
        bad = frame.loc[invalid_keys, "db_key"].tolist()
        raise ValueError(f"Invalid db_key values: {bad}")

    duplicate_keys = frame["db_key"].duplicated(keep=False)
    if duplicate_keys.any():
        bad = frame.loc[duplicate_keys, "db_key"].tolist()
        raise ValueError(f"Duplicate db_key values: {bad}")

    for column, lower, upper in (
        ("latitude", -90.0, 90.0),
        ("longitude", -180.0, 180.0),
    ):
        values = pd.to_numeric(frame[column], errors="coerce")
        invalid = values.isna() | ~values.map(math.isfinite) | ~values.between(lower, upper)
        if invalid.any():
            rows = list(frame.index[invalid])
            raise ValueError(f"Invalid {column} values at input rows {rows}")
        frame[column] = values.astype(float)

    if "sequence" in frame.columns:
        sequence = pd.to_numeric(frame["sequence"], errors="coerce")
        invalid_sequence = (
            sequence.isna()
            | ~sequence.map(math.isfinite)
            | sequence.mod(1).ne(0)
        )
        if invalid_sequence.any():
            rows = list(frame.index[invalid_sequence])
            raise ValueError(f"Invalid sequence values at input rows {rows}")
        if sequence.duplicated(keep=False).any():
            duplicates = sequence[sequence.duplicated(keep=False)].tolist()
            raise ValueError(f"Duplicate sequence values: {duplicates}")
        frame["sequence"] = sequence.astype("int64")
        frame = frame.sort_values("sequence", kind="stable")
    else:
        frame.insert(0, "sequence", range(1, len(frame) + 1))

    return frame.reset_index(drop=True)


In [ ]:
points = validate_profile_points(raw_points)
print(f"Validated {len(points):,} ordered points.")
display(points)

## 6. Query elevations in sequential batches

The service accepts at most 500 points in one request. This client can process a longer profile by sending smaller batches sequentially and reassembling the results. It checks the returned keys and order before attaching results to the input table.

In [ ]:
def response_error_text(response: requests.Response) -> str:
    try:
        return json.dumps(response.json(), indent=2)
    except ValueError:
        return response.text[:1000]


def request_profile_elevations(
    ordered_points: pd.DataFrame,
    *,
    base_url: str,
    units: str,
    batch_size: int,
    timeout_seconds: int,
    pause_seconds: float,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    if not 1 <= batch_size <= 500:
        raise ValueError("batch_size must be between 1 and 500.")

    endpoint = f"{base_url.rstrip('/')}/api/v1/elevations"
    all_results: list[dict[str, Any]] = []
    expected_metadata: dict[str, Any] | None = None
    total_batches = math.ceil(len(ordered_points) / batch_size)

    with requests.Session() as session:
        session.headers.update({"User-Agent": "lcg-topographic-profile-notebook/0.1"})

        for batch_number, start in enumerate(
            range(0, len(ordered_points), batch_size),
            start=1,
        ):
            batch = ordered_points.iloc[start : start + batch_size]
            payload = {
                "units": units,
                "points": batch[["db_key", "latitude", "longitude"]].to_dict(
                    orient="records"
                ),
            }
            print(
                f"Requesting batch {batch_number}/{total_batches} "
                f"with {len(batch):,} points..."
            )
            response = session.post(endpoint, json=payload, timeout=timeout_seconds)
            if not response.ok:
                raise RuntimeError(
                    f"Elevation request failed with HTTP {response.status_code}:\n"
                    f"{response_error_text(response)}"
                )

            document = response.json()
            batch_results = document.get("results")
            if not isinstance(batch_results, list):
                raise RuntimeError("Elevation response does not contain a results list.")

            returned_keys = [result.get("db_key") for result in batch_results]
            expected_keys = batch["db_key"].tolist()
            if returned_keys != expected_keys:
                raise RuntimeError(
                    "Service result keys or ordering did not match the request batch."
                )

            metadata = {key: value for key, value in document.items() if key != "results"}
            if expected_metadata is None:
                expected_metadata = metadata
            elif metadata != expected_metadata:
                raise RuntimeError("Service metadata changed between profile batches.")

            all_results.extend(batch_results)
            if batch_number < total_batches and pause_seconds > 0:
                time.sleep(pause_seconds)

    result_frame = pd.DataFrame(all_results)
    if result_frame["db_key"].tolist() != ordered_points["db_key"].tolist():
        raise RuntimeError("Reassembled result order does not match the input profile.")

    profile = ordered_points.copy().reset_index(drop=True)
    for column in ("elevation", "status", "message"):
        profile[column] = result_frame[column]
    profile["elevation"] = pd.to_numeric(profile["elevation"], errors="coerce")

    if expected_metadata is None:
        raise RuntimeError("No elevation batches were requested.")
    return profile, expected_metadata


In [ ]:
profile, service_metadata = request_profile_elevations(
    points,
    base_url=SERVICE_BASE_URL,
    units=ELEVATION_UNITS,
    batch_size=BATCH_SIZE,
    timeout_seconds=REQUEST_TIMEOUT_SECONDS,
    pause_seconds=PAUSE_BETWEEN_BATCHES_SECONDS,
)

print("Service metadata:")
display(pd.Series(service_metadata, name="value").to_frame())
display(profile)

## 7. Calculate distance along the profile

Coordinates remain EPSG:4326 longitude/latitude. `pyproj.Geod` calculates each segment on the WGS 84 ellipsoid, avoiding the incorrect assumption that one decimal degree is a fixed number of feet or meters. Segment slope is calculated only when both neighboring elevations are available.

In [ ]:
def add_profile_distances(profile_frame: pd.DataFrame, elevation_units: str) -> pd.DataFrame:
    geod = Geod(ellps="WGS84")
    frame = profile_frame.copy()
    segment_distances_m = [0.0]

    for previous, current in zip(
        frame.iloc[:-1].itertuples(index=False),
        frame.iloc[1:].itertuples(index=False),
        strict=True,
    ):
        _, _, distance_m = geod.inv(
            previous.longitude,
            previous.latitude,
            current.longitude,
            current.latitude,
        )
        segment_distances_m.append(float(distance_m))

    frame["segment_distance_m"] = segment_distances_m
    frame["distance_m"] = frame["segment_distance_m"].cumsum()
    frame["distance_km"] = frame["distance_m"] / 1000.0
    frame["distance_miles"] = frame["distance_m"] / 1609.344

    elevation_to_meters = 0.3048 if elevation_units == "feet" else 1.0
    elevation_change_m = frame["elevation"].diff() * elevation_to_meters
    frame["segment_slope_percent"] = (
        elevation_change_m / frame["segment_distance_m"] * 100.0
    )
    frame.loc[frame.index[0], "segment_slope_percent"] = pd.NA
    return frame


profile = add_profile_distances(profile, ELEVATION_UNITS)
display(
    profile[
        [
            "sequence",
            "db_key",
            "distance_miles",
            "distance_km",
            "elevation",
            "status",
            "segment_slope_percent",
        ]
    ]
)

## 8. Review profile quality

A `no_data` result is retained as a missing elevation. Investigate those coordinates before deciding whether interpolation is scientifically appropriate. This notebook never replaces missing values silently.

In [ ]:
valid_elevations = profile.loc[profile["status"].eq("success"), "elevation"].dropna()
summary = {
    "input_points": len(profile),
    "successful_points": int(profile["status"].eq("success").sum()),
    "no_data_points": int(profile["status"].eq("no_data").sum()),
    "profile_length_miles": float(profile["distance_miles"].iloc[-1]),
    "profile_length_km": float(profile["distance_km"].iloc[-1]),
    "minimum_elevation": float(valid_elevations.min()) if not valid_elevations.empty else None,
    "maximum_elevation": float(valid_elevations.max()) if not valid_elevations.empty else None,
    "elevation_units": ELEVATION_UNITS,
}
display(pd.Series(summary, name="value").to_frame())

no_data_rows = profile.loc[profile["status"].ne("success")]
if no_data_rows.empty:
    print("All profile points returned elevations.")
else:
    print("Review these non-success results:")
    display(no_data_rows[["sequence", "db_key", "latitude", "longitude", "status", "message"]])

## 9. Plot the topographic profile

The horizontal and vertical units are labeled independently. Missing elevations remain gaps in the plotted line. Vertical exaggeration is not applied automatically because the apparent slope depends on the figure's physical aspect ratio.

In [ ]:
x_column = "distance_miles" if HORIZONTAL_UNITS == "miles" else "distance_km"
x_label = (
    "Distance along profile (miles)"
    if HORIZONTAL_UNITS == "miles"
    else "Distance along profile (km)"
)
y_label = "Elevation (international feet)" if ELEVATION_UNITS == "feet" else "Elevation (meters)"

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.plot(
    profile[x_column],
    profile["elevation"],
    color="#176B5B",
    linewidth=2.0,
    marker="o",
    markersize=4.5,
)
ax.set_title("LCG 3DEP topographic profile")
ax.set_xlabel(x_label)
ax.set_ylabel(y_label)
ax.grid(True, color="#D9E2E1", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)

successful = profile.loc[profile["status"].eq("success") & profile["elevation"].notna()]
if not successful.empty:
    for row in (successful.iloc[0], successful.iloc[-1]):
        ax.annotate(
            str(row["db_key"]),
            (row[x_column], row["elevation"]),
            xytext=(5, 7),
            textcoords="offset points",
            fontsize=8,
        )

fig.tight_layout()
plt.show()

## 10. Export reviewable results

CSV retains the complete ordered table. GeoJSON contains one profile LineString and one Point feature per sample in WGS 84 longitude/latitude order. Metadata is saved separately so downstream users can identify the elevation source and vertical reference.

In [ ]:
def json_scalar(value: Any) -> Any:
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        return value.item()
    return value


OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
csv_path = OUTPUT_DIRECTORY / "topographic_profile_results.csv"
geojson_path = OUTPUT_DIRECTORY / "topographic_profile.geojson"
plot_path = OUTPUT_DIRECTORY / "topographic_profile.png"
metadata_path = OUTPUT_DIRECTORY / "topographic_profile_metadata.json"

profile.to_csv(csv_path, index=False)
fig.savefig(plot_path, dpi=180, bbox_inches="tight")

line_coordinates = [
    [float(row.longitude), float(row.latitude)]
    for row in profile.itertuples(index=False)
]
features: list[dict[str, Any]] = [
    {
        "type": "Feature",
        "properties": {"feature_type": "profile_line", **service_metadata},
        "geometry": {"type": "LineString", "coordinates": line_coordinates},
    }
]

for record in profile.to_dict(orient="records"):
    longitude = float(record.pop("longitude"))
    latitude = float(record.pop("latitude"))
    properties = {key: json_scalar(value) for key, value in record.items()}
    properties["feature_type"] = "profile_point"
    features.append(
        {
            "type": "Feature",
            "properties": properties,
            "geometry": {"type": "Point", "coordinates": [longitude, latitude]},
        }
    )

geojson = {"type": "FeatureCollection", "features": features}
with geojson_path.open("w", encoding="utf-8") as stream:
    json.dump(geojson, stream, indent=2, allow_nan=False)
    stream.write("\n")

with metadata_path.open("w", encoding="utf-8") as stream:
    json.dump(service_metadata, stream, indent=2, allow_nan=False)
    stream.write("\n")

for path in (csv_path, geojson_path, plot_path, metadata_path):
    print(path)

## 11. Next controlled tests

After the synthetic profile succeeds:

1. Replace `INPUT_PATH` with a small reviewed export from the ArcGIS Pro line-to-points tool.
2. Confirm that its geometries were transformed to EPSG:4326 and that `sequence` follows the intended line direction.
3. Compare several returned elevations with independent map or desktop-GIS inspection.
4. Increase the point count gradually while retaining the sequential batching and quality-control review.
5. Decide which fields and plots should become part of the ArcGIS tool and public web interface.